[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soumyaroy799/SUIT_solar_workshops/blob/main/workshops/USO_Vikram_2026_Oct/notebooks/coalignment.ipynb)

### First we will set the environment, download the data necessary. You do not need to really care about this, the necessary important parts start from the cell after this.

In [1]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/soumyaroy799/SUIT_solar_workshops.git"

if IN_COLAB:

    ROOT = Path("SUIT_solar_workshops")

    if not ROOT.exists():

        print("Cloning workshop repository...")

        subprocess.run(
            ["git", "clone", REPO_URL],
            check=True,
        )

    ROOT = ROOT.resolve()

else:

    ROOT = Path.cwd().resolve()

    while ROOT.name != "SUIT_solar_workshops":

        if ROOT.parent == ROOT:

            raise RuntimeError(
                "Please run inside "
                "SUIT_solar_workshops"
            )

        ROOT = ROOT.parent

import sys

sys.path.insert(
    0,
    str(ROOT),
)

In [2]:
from workshop_setup import setup_workshop

WORKSHOP_DIR, DATA_DIR = setup_workshop(ROOT)

Qt backend enabled
Workshop data already available (19 FITS files found).
Point 1: (418.0, 572.1)
Point 2: (577.2, 424.4)

ROI selected.
Run create_submap()


In [3]:
import numpy as np
import pickle
import time
import matplotlib.pyplot as plt
from astropy.io import fits
from glob import glob
from sunpy.map import Map
from datetime import datetime
import matplotlib.patches as patches
from matplotlib.widgets import RectangleSelector
import astropy.units as u
from astropy.coordinates import SkyCoord
import matplotlib.patches as patches

from sunkit_image.coalignment import (
    calculate_match_template_shift as cal_shift
)
from sunkit_image.coalignment import apply_shifts

from sunpy.coordinates import get_horizons_coord
from sunpy.map import get_observer_meta

import warnings
warnings.filterwarnings("ignore")

from copy import deepcopy

In [4]:
def co_align(map_sequence, template, layer_index=0, clip=False):
    """
    Co-align a map sequence to a template.

    Parameters
    ----------
    map_sequence : sequence of sunpy.map.Map
    template : sunpy.map.Map
        Template used for cross-correlation.
    layer_index : int, optional
        Reference layer in the sequence.
    clip : bool, optional
        Clip edge pixels affected by shifting.

    Returns
    -------
    list
        Co-aligned maps.
    """

    shifts = cal_shift(
        map_sequence,
        layer_index=layer_index,
        template=template
    )

    plate_scale = map_sequence[0].scale[0]

    return apply_shifts(
        map_sequence,
        xshift=-shifts["x"] / plate_scale,
        yshift=-shifts["y"] / plate_scale,
        clip=clip,
    )

def start_roi_selection(sunpy_map):
    """
    Display a map and collect two mouse clicks.

    Returns
    -------
    fig, ax, roi
    """

    roi = {
        "clicks": [],
    }

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection=sunpy_map)

    sunpy_map.plot(axes=ax)

    ax.set_title(
        "Click two opposite corners of the ROI"
    )

    def onclick(event):

        if (
            event.xdata is None
            or event.ydata is None
        ):
            return

        roi["clicks"].append(
            (event.xdata, event.ydata)
        )

        print(
            f"Point {len(roi['clicks'])}: "
            f"({event.xdata:.1f}, "
            f"{event.ydata:.1f})"
        )

        if len(roi["clicks"]) == 2:

            (x1, y1), (x2, y2) = roi["clicks"]

            xmin = min(x1, x2)
            xmax = max(x1, x2)

            ymin = min(y1, y2)
            ymax = max(y1, y2)

            rect = patches.Rectangle(
                (xmin, ymin),
                xmax - xmin,
                ymax - ymin,
                fill=False,
                linewidth=2,
                color="red",
            )

            ax.add_patch(rect)

            fig.canvas.draw_idle()

            print(
                "\nROI selected."
                "\nRun create_submap()"
            )

    fig.canvas.mpl_connect(
        "button_press_event",
        onclick,
    )

    plt.show()

    return fig, ax, roi

def create_submap(
    sunpy_map,
    roi,
):
    """
    Create a submap from ROI clicks.
    """

    if len(roi["clicks"]) != 2:

        raise RuntimeError(
            "ROI not yet selected."
        )

    (x1, y1), (x2, y2) = roi["clicks"]

    xmin = min(x1, x2)
    xmax = max(x1, x2)

    ymin = min(y1, y2)
    ymax = max(y1, y2)

    bottom_left = u.Quantity(
        [xmin, ymin],
        u.pixel,
    )

    top_right = u.Quantity(
        [xmax, ymax],
        u.pixel,
    )

    return sunpy_map.submap(
        bottom_left=bottom_left,
        top_right=top_right,
    )

## Loading SUIT Images

### In this section, we load a sequence of SUIT Region-of-Interest (RoI) images and create a SunPy `MapSequence`. We will use these images to demonstrate image coalignment techniques.

#### SUIT (Solar Ultraviolet Imaging Telescope) is one of the remote-sensing payloads onboard Aditya-L1. It observes the Sun in 11 ultraviolet passbands (8 narrow-band and 3 broad-band filters) covering the wavelength range 200–400 nm. SUIT provides both full-disk and Region-of-Interest observations, allowing high-cadence studies of solar activity in selected regions. :chatgpt-content-reference{index="0"}

#### The example data for this workshop are provided in:

```text
workshops/USO_Vikram_2026_Oct/data/
```

#### The following cell loads all FITS files in this directory into a SunPy `MapSequence`.


### For a detailed description of SUIT, its observing modes, filters, and science objectives, see [Tripathi et al. (2025)](https://link.springer.com/article/10.1007/s11207-025-02423-1)

In [5]:
files = sorted(DATA_DIR.glob("*.fits"))

smaps = Map([str(f) for f in files], sequence=True)

smaps.peek()

 ## Making a template to colaign the maps.
 Cautions while making a template;
   1. Pick a region which does not change and appears similar in all the images in your map sequence
   2. Always pick a region which is smaller than the size of the map.
   3. Sunspots are a good choice if you are trying to coalign different filters as they will appear in all filters with fairly similar structure.

### In the next cell you will select a region from one of the maps to create a submap. In the next cell, the submap will be created and plotted. If you do not like the selection of the template, you can reslect by running the cell.

In [6]:
index_nb3 = 1

fig, ax, roi = start_roi_selection(
    smaps[index_nb3]
)

In [7]:
template = create_submap(
    smaps[index_nb3],
    roi,
)

template.peek()

In [8]:
coaligned_maps = co_align(smaps[:7],template=template,layer_index=index_nb3)
coaligned_maps.peek()

### If the maps are not coaligned properly, go back and select a new template. Have Fun!
### These maps can also be saved in a single file, so that you don't have to do this process again.
#### The same can be done for files from other observatories also e.g. AIA, EUI etc

In [13]:
CACHE_DIR = WORKSHOP_DIR / "cache"
CACHE_DIR.mkdir(parents=True,exist_ok=True,)

with open(CACHE_DIR / "mapsequence.pkl","wb",) as f:
    pickle.dump(coaligned_maps,f,)

#### You can later load the pickled map sequence as you see fit!

In [14]:
with open(CACHE_DIR / "mapsequence.pkl","rb",) as f:
    loaded_maps = pickle.load(f)

In [15]:
loaded_maps.peek()